In [1]:
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from keras.models import Model, Sequential
from keras.layers import GRU, Input, Dense, TimeDistributed, Activation, RepeatVector, Bidirectional,LSTM, Dropout, Embedding
from keras.optimizers import Adam
from keras.losses import sparse_categorical_crossentropy
from keras.callbacks import ModelCheckpoint

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
from tensorflow.keras.models import load_model

model = load_model("/content/drive/MyDrive/Colab Notebooks/UIT/model_1.h5")

In [7]:
import pandas as pd
import numpy as np

In [8]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/UIT/eng_-french.csv")

In [9]:
df.head()

,English words/sentences,French words/sentences
0,Hi.,Salut!
1,Run!,Cours !
2,Run!,Courez !
3,Who?,Qui ?
4,Wow!,Ça alors !


In [10]:
df_eng=df["English words/sentences"]
df_fr=df["French words/sentences"]

In [11]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [12]:
def tokenize(x):

  tokenizer = Tokenizer()
  tokenizer.fit_on_texts(x)
  return tokenizer.texts_to_sequences(x), tokenizer

In [13]:
def pad(x, length=0):
  return pad_sequences(x, maxlen=40, padding = "post")

In [14]:
def preprocessing(x, y):

  preprocess_x, x_tokens = tokenize(x)
  preprocess_y, y_tokens = tokenize(y)

  preprocess_x = pad(preprocess_x)
  preprocess_y = pad(preprocess_y)

  preprocess_y = preprocess_y.reshape(*preprocess_y.shape, 1)

  return preprocess_x, preprocess_y, x_tokens, y_tokens

In [15]:
preprocess_english_sentences, preprocess_french_sentences, english_tokenizer, french_tokenizer = preprocessing(df_eng, df_fr)

In [16]:
preprocess_english_sentences

array([[2818,    0,    0, ...,    0,    0,    0],
       [ 429,    0,    0, ...,    0,    0,    0],
       [ 429,    0,    0, ...,    0,    0,    0],
       ...,
       [ 621,    6,   97, ...,    0,    0,    0],
       [7819, 2915,   32, ...,   97,  810, 4250],
       [  30,   23, 5965, ...,    5, 1124, 1480]], dtype=int32)

In [17]:
max_english_sequence_length = preprocess_english_sentences.shape[1]
max_french_sequence_length = preprocess_french_sentences.shape[1]
english_vocab_size = len(english_tokenizer.word_index)
french_vocab_size = len(french_tokenizer.word_index)

print("Max English sentence length:", max_english_sequence_length)
print("Max French sentence length:", max_french_sequence_length)
print("English vocabulary size:", english_vocab_size)
print("French vocabulary size:", french_vocab_size)

Max English sentence length: 40
Max French sentence length: 40
English vocabulary size: 14531
French vocabulary size: 30660


In [18]:
def logits_to_text(logits, tokenizer):

    index_to_words = {id: word for word, id in tokenizer.word_index.items()}
    index_to_words[0] = ''

    return ' '.join([index_to_words[prediction] for prediction in np.argmax(logits, 1)])

In [19]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPUs available:", tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.19.0
GPUs available: []


In [ ]:
def bd_model(input_shape, output_sequence_length, english_vocab_size, french_vocab_size):

    learning_rate = 0.003

    # Build the layers
    model = Sequential()
    model.add(Embedding(french_vocab_size, 256, input_length=input_shape[1], input_shape=input_shape[1:]))
    model.add(Bidirectional(GRU(256, return_sequences=True)))
    model.add(TimeDistributed(Dense(1024, activation='relu')))
    model.add(Dropout(0.5))
    model.add(TimeDistributed(Dense(english_vocab_size, activation='softmax')))

    # Compile model
    model.compile(loss=sparse_categorical_crossentropy,
                  optimizer=Adam(learning_rate),
                  metrics=['accuracy'])
    return model

In [23]:
tmp_x = pad(preprocess_french_sentences, preprocess_french_sentences.shape[1])
tmp_x = tmp_x.reshape((-1, preprocess_french_sentences.shape[-2]))

# Train
model = bd_model(
    tmp_x.shape,
    preprocess_english_sentences.shape[1],
    len(english_tokenizer.word_index)+1,
    len(french_tokenizer.word_index)+1)

model.summary()

model.fit(tmp_x, preprocess_english_sentences, batch_size=64, epochs=5, validation_split=0.2)

model.save("/content/drive/MyDrive/Colab Notebooks/UIT/model_2.h5")

In [27]:
i=100120
print("Prediction:")
print(logits_to_text(model.predict(tmp_x[[i]])[0], english_tokenizer))
print("\nCorrect Translation:")
print(df_eng[i])
print("\nOriginal text:")
print(df_fr[i])

Prediction:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 165ms/step
we're safe here here                                    

Correct Translation:
We're safe in here, aren't we?

Original text:
Nous sommes en sécurité ici, n'est-ce pas ?


In [29]:
i=100097
print("Prediction:")
print(logits_to_text(model.predict(tmp_x[[i]])[0], english_tokenizer))
print("\nCorrect Translation:")
print(df_eng[i])
print("\nOriginal text:")
print(df_fr[i])

Prediction:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 311ms/step
we're going to make some examples                                  

Correct Translation:
We're going to run some tests.

Original text:
Nous allons procéder à quelques tests.
